In [1]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
# output_file = "llama3.1_result.jsonl"
output_file = "llama8b_r1_result.jsonl"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

def load_model(model_name, device="auto", dtype=torch.bfloat16):
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device,
    )
    return model, tokenizer

model, tokenizer = load_model(model_name)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
# Text Generation
if "Llama" in model_name:
    BOS = 128000
    USER = 128011
    ASSISTANT = 128012
    NEWLINE = 198
    THINK_START = 128013
    THINK_END = 128014
    EOS = 128001
elif "Qwen" in model_name:
    BOS = 151646
    USER = 151644
    ASSISTANT = 151645
    NEWLINE = 198
    THINK_START = 151648
    THINK_END = 151649
    EOS = 151643
else:
    raise ValueError(f"Unknown tokens for model {model_name}")

In [18]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

def pprint(text):
    """Pretty print the model's generated text using rich."""
    console = Console(width=100)
    
    
    # Create markdown and display in a panel
    #md = Markdown(text.strip())
    console.print(Panel(text, border_style="blue"))


ModuleNotFoundError: No module named 'rich'

In [ ]:
# load math 500 dataset: HuggingFaceH4/MATH-500
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/MATH-500")

if "R1" in model_name:
    def prompt_from_example(example, tokenizer):
        user_message = example['problem']
        math_suffix = " Please reason step by step, and put your final answer within \\boxed{}."
        toks = [BOS] + [USER] + tokenizer.encode(user_message+math_suffix, add_special_tokens=False) + [ASSISTANT] + [THINK_START] + [NEWLINE]
        #toks = [BOS] + tokenizer.encode(user_message+math_suffix, add_special_tokens=False) + [THINK_START] + [NEWLINE]
        return toks, tokenizer.decode(toks, skip_special_tokens=False)
else:
    def prompt_from_example(example, tokenizer):
        user_message = example['problem']
        math_suffix = " Please reason step by step, and put your final answer within \\boxed{}."
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_message + math_suffix},
        ]
        toks = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        return toks, tokenizer.decode(toks, skip_special_tokens=False)

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

In [20]:
toks, prompt = prompt_from_example(dataset['test'][0], tokenizer)
answer = dataset['test'][0]['answer']

In [21]:
tokenizer

LlamaTokenizerFast(name_or_path='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', vocab_size=128000, model_max_length=16384, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>', 'pad_token': '<｜end▁of▁sentence｜>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	128000: AddedToken("<｜begin▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<｜end▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=T

In [24]:
# generate a continuation of prompt using huggingface text generation pipeline
from transformers import pipeline

if "R1" in model_name:
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        max_length=32768,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=EOS
    )
else:
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        do_sample=True,
        temperature=0.6,
        max_length=15000,
        pad_token_id=128002,
        eos_token_id=tokenizer.eos_token_id
    )

Device set to use cuda:0


In [ ]:
out = generator(prompt)
pprint(out[0]['generated_text'])

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


<｜begin▁of▁sentence｜><｜User｜>Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$ Please reason step by step, and put your final answer within \boxed{}.<｜Assistant｜><think>
Okay, so I need to convert the rectangular coordinate (0, 3) to polar coordinates. Hmm, polar coordinates are represented as (r, θ), where r is the distance from the origin and θ is the angle from the positive x-axis. I remember there are formulas to convert from rectangular to polar coordinates. Let me try to recall them.

First, I think r is calculated using the formula r = sqrt(x² + y²). So, for the point (0, 3), x is 0 and y is 3. Plugging those in, r would be sqrt(0² + 3²) which is sqrt(0 + 9) so sqrt(9) is 3. Okay, so r is 3. That seems straightforward.

Now, θ is a bit trickier. I remember θ is calculated using the arctangent function, specifically θ = arctan(y / x). But wait, in this case, x is 0. Hmm

In [ ]:
import re
from math500.grader import grade_answer

if "R1" in model_name:
    def parse_answer(generated_text):
        matches = re.search("</think>", generated_text)
        if matches is None:
            return ""
        generated_answer = generated_text[matches.end():]
        # in generated answer select the content of \boxed{}
        matches = re.search("\\\\boxed{", generated_answer)
        if matches is None:
            return ""
        generated_answer = generated_answer[matches.end():]
        # search all the way to the end of the string   }
        reversed = generated_answer[::-1]
        matches = re.search("}", reversed)
        if matches is None:
            return ""
        generated_answer = generated_answer[:len(generated_answer) - matches.start()]
        return generated_answer
else:
    def parse_answer(generated_text):
        end_idx = generated_text.find("<|start_header_id|>assistant<|end_header_id|>")
        generated_answer = generated_text[end_idx:]
        matches = re.search("\\\\boxed{", generated_answer)
        if matches is None:
            return ""
        generated_answer = generated_answer[matches.end():]
        # search all the way to the end of the string   }
        reversed = generated_answer[::-1]
        matches = re.search("}", reversed)
        if matches is None:
            return ""
        generated_answer = generated_answer[:len(generated_answer) - matches.start()]
        return generated_answer
parsed_answer = parse_answer(out[0]['generated_text'])
print(parsed_answer)
print(parsed_answer, answer)
grade_answer(parsed_answer, answer)

In [ ]:
# evaluate question by question and store results in both dataframe and jsonl file
import pandas as pd
import json
from tqdm import tqdm

df = pd.DataFrame(columns=['problem', 'answer', 'generated_answer', 'correct'])
n_correct = 0
for idx, example in tqdm(enumerate(dataset['test'])):
    toks, prompt = prompt_from_example(example, tokenizer)
    out = generator(prompt)
    generated_answer = parse_answer(out[0]['generated_text'])
    example['parsed_answer'] = generated_answer
    example['correct'] = grade_answer(generated_answer, example['answer'])
    example['generated_text'] = out[0]['generated_text']
    n_correct += int(example['correct'])
    print(n_correct / (idx + 1))

    # add to both dataframe and jsonl file
    df = pd.concat([df, pd.DataFrame([example])], ignore_index=True)
    
    # Write to jsonl file immediately after each example
    with open(output_file, 'a') as f:
        f.write(json.dumps(example) + '\n')
        f.flush() # Ensure it's written to disk
        
    #if idx > 3:
    #    break
